In [1]:
# Imports and Environment Setup
import pandas as pd
import numpy as np
import os

# Create the data directory if it doesn't exist yet
if not os.path.exists('data'):
    os.makedirs('data')

# Define paths (Relative to your project folder)
books_path = "C:\\Goodreads-NYT-Bestseller-Predictor\\data\\goodreads_books.json.gz"
reviews_path = "C:\\Goodreads-NYT-Bestseller-Predictor\\data\\goodreads_reviews_dedup.json.gz"
nyt_path = "C:\\Goodreads-NYT-Bestseller-Predictor\\data\\bestsellers.csv"

In [2]:
# Optimized Data Loading Function
def load_data_fast(path, nrows=None):
    print(f"Reading {path}...")
    # 'lines=True' is required for JSONL format
    # 'compression=infer' handles the .gz automatically and quickly
    return pd.read_json(path, lines=True, compression='infer', nrows=nrows)

In [3]:
# Ingesting the Datasets
# 1. Load NYT (Smallest file)
df_nyt = pd.read_csv(nyt_path)
print(f"NYT Bestseller data loaded: {len(df_nyt)} records.")

# 2. Load and Filter Books using Chunks
# This prevents MemoryError by only keeping books from 2010-2019 and English language
books_chunks = []
chunk_size = 100000 # Process 100k rows at a time

print("Starting chunked processing...")

# We use pd.read_json with chunksize to create an iterator
reader = pd.read_json(books_path, lines=True, compression='infer', chunksize=chunk_size)

for i, chunk in enumerate(reader):
    # 1. Filter for English immediately
    chunk['language_code'] = chunk['language_code'].fillna('').astype(str).str.lower()
    chunk = chunk[chunk['language_code'].str.startswith('en')]
    
    # 2. Filter for Year 2010-2019 (Your project scope)
    chunk['publication_year'] = pd.to_numeric(chunk['publication_year'], errors='coerce')
    chunk = chunk[chunk['publication_year'].between(2010, 2019)]
    
    # 3. Only keep the columns we actually need for the report
    keep_cols = [
        'book_id', 'title', 'authors', 'publisher', 'publication_year', 
        'average_rating', 'ratings_count', 'text_reviews_count', 'description'
    ]
    books_chunks.append(chunk[keep_cols])
    
    if (i + 1) % 5 == 0:
        print(f"Processed {(i + 1) * chunk_size:,} raw rows...")

# Combine the filtered chunks into one manageable DataFrame
df_books = pd.concat(books_chunks, ignore_index=True)
print(f"\nFinal filtered Books dataset: {len(df_books):,} records.")

NYT Bestseller data loaded: 61430 records.
Starting chunked processing...
Processed 500,000 raw rows...
Processed 1,000,000 raw rows...
Processed 1,500,000 raw rows...
Processed 2,000,000 raw rows...

Final filtered Books dataset: 458,516 records.


In [4]:
# Data Quality & Missing Value Analysis
# True missing logic
def true_missing(df):
    """
    Identifies hidden nulls like '[]', 'none', or 'NULL' 
    which standard isnull() functions often miss in JSON data.
    """
    return (df.isin(['', 'none', 'nan', 'NULL', '[]', '{}'])
             .sum() + df.isnull().sum()) / len(df) * 100

print("Data Quality Analysis (Percentage of Missing Values):")
print(true_missing(df_books).round(2))

Data Quality Analysis (Percentage of Missing Values):
book_id                0.00
title                  0.00
authors                0.00
publisher             12.39
publication_year       0.00
average_rating         0.00
ratings_count          0.00
text_reviews_count     0.00
description            3.41
dtype: float64


In [5]:
# Data Filtering and Checkpoint Save
# Define essential columns for the feature engineering stage
keep_cols = [
    'book_id', 'title', 'authors', 'publisher', 'publication_year', 
    'average_rating', 'ratings_count', 'text_reviews_count', 'description'
]

# Save a smaller "interim" CSV to skip the heavy JSON loading in the next session
df_books[keep_cols].to_csv('data/interim_books.csv', index=False)

print("Notebook 1 process complete.")
print("Checkpoint saved to: 'data/interim_books.csv'")

Notebook 1 process complete.
Checkpoint saved to: 'data/interim_books.csv'
